# Cochin MS Oo.1.32 — PDF → OSIS Converter (James)

Converts the PDF document into three OSIS XML files:
- `*_hebrew.osis` — Hebrew text only
- `*_hebrew_commented.osis` — Hebrew text with inline footnotes
- `*_translation.osis` — English translation with inline footnotes

**PDF structure per verse (JAS):**
1. `James N:V` or `James N:V (Cochin N:AV)` or `James N:V (KJV N:AV)` header block (size ~16.1)
2. `Image from Cochin Hebrew MS Oo.1.32` caption (skip)
3. `Transcription:` label block (Hebrew may be inline in same block)
4. Hebrew text block(s) (size ~13.9)
5. `English Translation: ...` block with English translation
6. Footnotes: blocks starting with `digit space capital-letter`

**Alt numbering (two conventions):**
- `James 1:22 (Cochin 1:21)` → primary=KJV/standard, alt=Cochin MS
- `James 2:16 (KJV 2:17)` → primary=Cochin MS, alt=KJV/standard

**Empty verses:** `James 1:21` → Transcription block says "This verse does not exist..."

In [ ]:
import fitz  # PyMuPDF
import re
import pathlib
from lxml import etree
from collections import defaultdict

# ── Config ──────────────────────────────────────────────────────────────────
PDF_PATH = pathlib.Path("../data/00_source_files/MS_Cochin_Oo.1.32_JAS_ProjectTruthMinistries.pdf")
OUT_DIR  = pathlib.Path("../data/01_osis")
STEM     = "Cochin_MS_Oo.1.32_JAS"

# Content starts at page 11 (0-indexed: 10). Pages 70-76 are back matter.
FIRST_CONTENT_PAGE = 10   # 0-indexed

print(f"PDF: {PDF_PATH}")
print(f"Output dir: {OUT_DIR}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

HEBREW_RE = re.compile(r'[\u05d0-\u05ea\u05b0-\u05c7\ufb1d-\ufb4e]')

def has_hebrew(text: str) -> bool:
    return bool(HEBREW_RE.search(text))

def hebrew_char_count(text: str) -> int:
    return sum(1 for c in text if '\u05b0' <= c <= '\u05ea' or '\ufb1d' <= c <= '\ufb4e')

def is_wide(x0: float, x1: float) -> bool:
    """Exclude single-character artifacts."""
    return (x1 - x0) > 50

def clean_hebrew(text: str) -> str:
    """Remove standalone footnote-marker digits scattered through Hebrew text."""
    text = re.sub(r'(?<!\S)\d+(?!\S)', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_footnote_number(text: str):
    """Return (n_str, rest) from a footnote block, or None."""
    m = re.match(r'^(\d+)\s+([A-Z\'""\u201c].+)', text.strip(), re.DOTALL)
    if m:
        return m.group(1), m.group(2).strip()
    return None

def block_texts(block):
    """Return (full_text, main_text) for a dict-mode block.
    main_text strips superscript spans (size < 85% of largest span).
    Use full_text for label matching; main_text only for verse header detection.
    """
    spans = [s for line in block.get("lines", []) for s in line.get("spans", [])]
    if not spans:
        return "", ""
    full_text = "".join(s["text"] for s in spans)
    sizes = [s["size"] for s in spans if s["text"].strip()]
    if not sizes:
        return full_text.strip(), full_text.strip()
    dominant = max(sizes)
    main_text = "".join(s["text"] for s in spans if s["size"] >= dominant * 0.85)
    return full_text.strip(), main_text.strip()

# ── Verse header parsing ─────────────────────────────────────────────────────
VERSE_HDR_RE = re.compile(
    r'James\s+(\d+):(\d+[ab]?)'
    r'(?:\s*\(([^)]+)\))?'
    r'\s*$'
)
PAREN_RE = re.compile(r'(?:Cochin|KJV)\s+(\d+):(\d+[ab]?)(?:-\d+[ab]?)?')

def parse_verse_header(main_text: str):
    """Parse a verse header from main-font text (superscripts stripped).
    Handles plain 'James N:V', '... (Cochin N:AV)', and '... (KJV N:AV)'.
    """
    main_text = main_text.strip()
    m = VERSE_HDR_RE.match(main_text)
    if not m:
        return None
    chapter = int(m.group(1))
    verse   = m.group(2)
    paren   = m.group(3) or ""

    alt_ch, alt_v = None, None
    if paren:
        pm = PAREN_RE.search(paren)
        if pm:
            alt_ch = int(pm.group(1))
            alt_v  = pm.group(2)

    return {
        "chapter": chapter,
        "verse":   verse,
        "alt_ch":  alt_ch,
        "alt_v":   alt_v,
        "empty":   False,
        "hebrew":  "",
        "english": "",
        "notes":   {},   # {footnote_n_str: text}
    }

# ── Block label regexes ──────────────────────────────────────────────────────
# Transcription label variants:
#   "Transcription:"
#   "Hebrew Transcription:"             (James 5:12, label at size 13.9 with Hebrew inline)
#   "Cochin Oo.1.32 Hebrew Transcription:"
TRANS_LABEL_RE = re.compile(r'^(?:Cochin Oo\.1\.32 )?(?:Hebrew )?Transcription:\s*')

# English translation label variants:
#   "English Translation:"
#   "Translation:"                      (James 5:20)
#   "Cochin Oo.1.32 English Translation:"
ENG_LABEL_RE = re.compile(r'^(?:Cochin Oo\.1\.32 )?(?:English )?Translation:\s*')

FOOTNOTE_START = re.compile(r'^\d+\s+[A-Z\'"\u201c]')

def is_footnote_block(text: str) -> bool:
    if not FOOTNOTE_START.match(text):
        return False
    if hebrew_char_count(text) > 3:
        return False
    return True

print("Helper functions defined.")

In [ ]:
# ── PDF parsing ──────────────────────────────────────────────────────────────
#
# State machine per verse:
#   HEB  → after verse header; looking for Transcription: label and Hebrew blocks
#   ENG  → after English Translation: block; collecting any continuation English
#
# The running page header "The Return Letter of James  Janice F. Baca" is at
# y=36-48 (y1=48). Using PAGE_RUNNING_HEADER_Y1=49 excludes it.
# Footer "Page N of 76" is at y=731-743.

VERSE_HEADER_RE        = re.compile(r'^James\s+\d+:\d+')
PAGE_RUNNING_HEADER_Y1 = 49
PAGE_FOOTER_Y0         = 728

def attach_pending_notes(verse, pending):
    if verse is not None:
        verse["notes"].update(pending)

doc = fitz.open(str(PDF_PATH))
print(f"Opened PDF: {len(doc)} pages")

verses        = []
state         = "SEEK"
current       = None
pending_notes = {}

for pg_idx in range(FIRST_CONTENT_PAGE, len(doc)):
    page = doc[pg_idx]
    d    = page.get_text("dict")
    blocks = sorted(d["blocks"], key=lambda b: b["bbox"][1])
    page_footnotes = {}

    for block in blocks:
        if block.get("type") != 0:
            continue
        x0, y0, x1, y1 = block["bbox"]

        # Skip running page header and footer
        if y1 < PAGE_RUNNING_HEADER_Y1 or y0 > PAGE_FOOTER_Y0:
            continue

        full_text, main_text = block_texts(block)
        if not full_text:
            continue

        # ── Footnote detection ──────────────────────────────────────────────
        if is_footnote_block(full_text):
            parts = re.split(r'(?m)(?=^\d+\s+[A-Z\'"\u201c])', full_text)
            for part in parts:
                part = part.strip()
                if not part:
                    continue
                result = extract_footnote_number(part)
                if result:
                    n_str, fn_text = result
                    page_footnotes[n_str] = fn_text
            continue

        # ── Verse header detection (use main_text to strip superscripts) ────
        # main_text strips the superscript footnote refs (e.g. "James 2:15²²" → "James 2:15")
        if VERSE_HEADER_RE.match(main_text):
            attach_pending_notes(current, pending_notes)
            pending_notes = {}
            parsed = parse_verse_header(main_text)
            if parsed:
                if current is not None:
                    verses.append(current)
                current = parsed
                state = "HEB"
            continue

        if current is None:
            continue

        # ── English Translation block ────────────────────────────────────────
        if ENG_LABEL_RE.match(full_text):
            body = ENG_LABEL_RE.sub('', full_text, count=1).strip()
            # Strip KJV comparison suffix some verses append
            for marker in ["  KJV", "\nKJV"]:
                if marker in body:
                    body = body[:body.index(marker)].strip()
            if body:
                current["english"] = (current["english"] + " " + body).strip()
            state = "ENG"
            continue

        # ── Transcription label block ────────────────────────────────────────
        if TRANS_LABEL_RE.match(full_text):
            if "does not exist" in full_text.lower():
                current["empty"] = True
            else:
                # Extract any Hebrew spans inline in this block
                # (happens when label and Hebrew share the same PDF block)
                spans = [s for line in block.get("lines", []) for s in line.get("spans", [])]
                heb_inline = " ".join(s["text"] for s in spans if has_hebrew(s["text"]))
                if heb_inline:
                    current["hebrew"] = (current["hebrew"] + " " + heb_inline).strip()
            # Stay in HEB — standalone Hebrew blocks may follow on same page
            continue

        # ── Content by state ─────────────────────────────────────────────────
        if state == "HEB":
            if has_hebrew(full_text) and is_wide(x0, x1):
                current["hebrew"] = (current["hebrew"] + " " + clean_hebrew(full_text)).strip()

        elif state == "ENG":
            if is_wide(x0, x1) and not has_hebrew(full_text):
                skip_prefixes = ("Image from Cochin",)
                if not any(full_text.startswith(p) for p in skip_prefixes):
                    current["english"] = (current["english"] + " " + full_text).strip()

    pending_notes.update(page_footnotes)

attach_pending_notes(current, pending_notes)
if current is not None:
    verses.append(current)

print(f"Parsed {len(verses)} verse records across {len(set(v['chapter'] for v in verses))} chapters.")
for v in verses[:3]:
    print(f"  Jas {v['chapter']}:{v['verse']} (alt:{v['alt_v']}) empty={v['empty']}")
    print(f"    heb: {v['hebrew'][:60]}")
    print(f"    eng: {v['english'][:60]}")

In [ ]:
# ── Inspect alt-numbered and empty verses ─────────────────────────────────────
from collections import Counter

alt_verses   = [v for v in verses if v['alt_v']]
empty_verses = [v for v in verses if v['empty']]
no_heb       = [v for v in verses if not v['empty'] and not v['hebrew'].strip()]

chapter_counts = Counter(v['chapter'] for v in verses)
print("=== Verse count by chapter ===")
for ch in sorted(chapter_counts):
    print(f"  Jas {ch}: {chapter_counts[ch]} verses")

print(f"\nAlt-numbered: {len(alt_verses)},  Empty: {len(empty_verses)},  No Hebrew: {len(no_heb)}")
print("\nAlt-numbered verses:")
for v in alt_verses:
    print(f"  Jas {v['chapter']}:{v['verse']}  alt {v['alt_ch']}:{v['alt_v']}")
print("\nEmpty verses:")
for v in empty_verses:
    print(f"  Jas {v['chapter']}:{v['verse']}")
if no_heb:
    print("\nVERSES MISSING HEBREW (investigate):")
    for v in no_heb:
        print(f"  Jas {v['chapter']}:{v['verse']}")

In [ ]:
# ── OSIS building helpers ─────────────────────────────────────────────────────

OSIS_NS  = "http://www.bibletechnologies.net/2003/OSIS/namespace"
XSI_NS   = "http://www.w3.org/2001/XMLSchema-instance"
ALT_NS   = "https://projecttruthministries.org/studies/cochin-james/"

SCHEMA_LOC = (f"{OSIS_NS} "
              "http://www.bibletechnologies.net/osisCore.2.1.1.xsd")

def make_osis_root(nsmap_extra=None):
    nsmap = {
        None:  OSIS_NS,
        "xsi": XSI_NS,
    }
    if nsmap_extra:
        nsmap.update(nsmap_extra)
    root = etree.Element(f"{{{OSIS_NS}}}osis", nsmap=nsmap)
    root.set(f"{{{XSI_NS}}}schemaLocation", SCHEMA_LOC)
    return root

def make_header_hebrew(parent):
    hdr = etree.SubElement(parent, f"{{{OSIS_NS}}}header")
    for osisWork, title, lang, extra in [
        ("MS.Oo.1.32_JAS_Hebrew", "James (Cochin MS Oo.1.32)", "he", [
            ("identifier", {"type": "OSIS"}, "MS.Oo.1.32_JAS_Hebrew"),
            ("refSystem", {}, "MS.Oo.1.32"),
            ("language", {}, "he"),
        ]),
        ("bible", "Referenced versification (standard)", "he", [
            ("identifier", {"type": "OSIS"}, "bible"),
            ("refSystem", {}, "StandardV11N"),
            ("language", {}, "he"),
        ]),
        ("MS.Oo.1.32", "James (Cochin MS Oo.1.32)", "he", [
            ("scope", {}, "JAS"),
            ("type", {"type": "x-manuscript"}, "Manuscript"),
            ("identifier", {"type": "shelfmark"}, "MS.Oo.1.32"),
            ("identifier", {"type": "URI"}, ALT_NS),
            ("publisher", {}, "Project Truth Ministries"),
            ("date", {"event": "original", "type": "ISO"}, "ca. 1730"),
            ("description", {}, (
                "The Cochin Hebrew New Testament manuscripts are significant 18th-century Hebrew "
                "versions of the New Testament currently housed at Cambridge. "
                "This file encodes the James portion of MS Oo.1.32, copied ca. 1730 in Cochin, India."
            )),
        ]),
    ]:
        w = etree.SubElement(hdr, f"{{{OSIS_NS}}}work", osisWork=osisWork)
        t = etree.SubElement(w, f"{{{OSIS_NS}}}title")
        t.text = title
        for tag, attrs, val in extra:
            el = etree.SubElement(w, f"{{{OSIS_NS}}}{tag}", **attrs)
            el.text = val

def make_header_translation(parent):
    hdr = etree.SubElement(parent, f"{{{OSIS_NS}}}header")
    w = etree.SubElement(hdr, f"{{{OSIS_NS}}}work", osisWork="MS.Oo.1.32_JAS_PTM")
    for tag, attrs, val in [
        ("title", {}, "Translation of James (Cochin MS Oo.1.32)"),
        ("scope", {}, "JAS"),
        ("type", {"type": "x-bible"}, "Edition"),
        ("creator", {"role": "trl"}, "Project Truth Ministries"),
        ("identifier", {"type": "URI"}, ALT_NS),
        ("contributor", {"role": "trc", "file-as": "Baca, Janice F."}, "Janice F. Baca"),
        ("date", {"event": "eversion", "type": "ISO"}, "2024"),
        ("rights", {}, "\u00a9 copyright 2024 Janice F. Baca"),
        ("description", {}, (
            "English translation of the Cochin MS Oo.1.32 Hebrew James. "
            "Translated by Janice F. Baca, 2024."
        )),
    ]:
        el = etree.SubElement(w, f"{{{OSIS_NS}}}{tag}", **attrs)
        el.text = val

def verse_osisID(chapter, verse):
    return f"Jas.{chapter}.{verse}"

def alt_osisID(alt_ch, alt_v):
    if alt_ch is None or alt_v is None:
        return None
    return f"Jas.{alt_ch}.{alt_v}"

print("OSIS helpers defined.")

In [ ]:
# ── Generate Hebrew OSIS (text only or with notes) ────────────────────────────

ALT_PREFIX = "{" + ALT_NS + "}"

def build_hebrew_osis(verses, with_notes=False):
    nsmap_extra = {"alt": ALT_NS}
    root = make_osis_root(nsmap_extra)

    work_id = "MS.Oo.1.32_JAS_Hebrew" + ("_Commented" if with_notes else "")
    osisText = etree.SubElement(root, f"{{{OSIS_NS}}}osisText",
                                osisIDWork=work_id,
                                osisRefWork="bible")
    osisText.set("{http://www.w3.org/XML/1998/namespace}lang", "he")

    make_header_hebrew(osisText)

    div_book = etree.SubElement(osisText, f"{{{OSIS_NS}}}div",
                                type="book", osisID="Jas")

    current_chapter = None
    chapter_el = None

    for v in verses:
        ch = v["chapter"]
        if ch != current_chapter:
            chapter_el = etree.SubElement(div_book, f"{{{OSIS_NS}}}chapter",
                                          osisID=f"Jas.{ch}")
            current_chapter = ch

        osisID = verse_osisID(ch, v["verse"])
        n_val  = v["verse"]
        alt_id = alt_osisID(v["alt_ch"], v["alt_v"])

        attrs = {"osisID": osisID, "n": n_val}
        if alt_id:
            attrs[f"{ALT_PREFIX}num"] = alt_id

        verse_el = etree.SubElement(chapter_el, f"{{{OSIS_NS}}}verse", **attrs)

        if not v["empty"]:
            hebrew_text = clean_hebrew(v["hebrew"])
            if with_notes and v["notes"]:
                verse_el.text = hebrew_text + " "
                for n_str, note_text in sorted(v["notes"].items(), key=lambda x: int(x[0])):
                    note_el = etree.SubElement(verse_el, f"{{{OSIS_NS}}}note",
                                               type="footnote", n=n_str)
                    note_el.text = note_text
                    note_el.tail = " "
            else:
                verse_el.text = hebrew_text

    return root

heb_tree      = build_hebrew_osis(verses, with_notes=False)
heb_commented = build_hebrew_osis(verses, with_notes=True)
print("Hebrew OSIS trees built.")

In [ ]:
# ── Generate Translation OSIS ─────────────────────────────────────────────────

def build_translation_osis(verses):
    root = make_osis_root()

    osisText = etree.SubElement(root, f"{{{OSIS_NS}}}osisText",
                                osisIDWork="JAS",
                                osisRefWork="bible")
    osisText.set("{http://www.w3.org/XML/1998/namespace}lang", "en")

    make_header_translation(osisText)

    current_chapter = None
    chapter_el = None

    for v in verses:
        ch = v["chapter"]
        if ch != current_chapter:
            chapter_el = etree.SubElement(osisText, f"{{{OSIS_NS}}}div",
                                          type="chapter",
                                          osisID=f"Jas.{ch}")
            current_chapter = ch

        osisID = verse_osisID(ch, v["verse"])
        verse_el = etree.SubElement(chapter_el, f"{{{OSIS_NS}}}verse",
                                    osisID=osisID)

        if not v["empty"] and v["english"]:
            eng_text = v["english"].replace("\n", " ").strip()
            if v["notes"]:
                verse_el.text = eng_text + " "
                for n_str, note_text in sorted(v["notes"].items(), key=lambda x: int(x[0])):
                    note_el = etree.SubElement(verse_el, f"{{{OSIS_NS}}}note",
                                               type="footnote", n=n_str)
                    note_el.text = note_text
                    note_el.tail = " "
            else:
                verse_el.text = eng_text

    return root

trans_tree = build_translation_osis(verses)
print("Translation OSIS tree built.")

In [ ]:
# ── Write output files ────────────────────────────────────────────────────────

outputs = [
    (heb_tree,      f"{STEM}_hebrew.osis"),
    (heb_commented, f"{STEM}_hebrew_commented.osis"),
    (trans_tree,    f"{STEM}_translation.osis"),
]

for tree, filename in outputs:
    out_path = OUT_DIR / filename
    tree_obj = etree.ElementTree(tree)
    tree_obj.write(
        str(out_path),
        xml_declaration=True,
        encoding="UTF-8",
        pretty_print=True,
    )
    size_kb = out_path.stat().st_size // 1024
    print(f"Wrote {out_path}  ({size_kb} KB)")

In [ ]:
# ── Verification ──────────────────────────────────────────────────────────────

chapter_counts = Counter(v['chapter'] for v in verses)
print("=== Verse count by chapter ===")
for ch in sorted(chapter_counts):
    print(f"  Jas {ch}: {chapter_counts[ch]} verses")

print(f"\nTotal: {len(verses)} verses")

print("\n=== Sample Jas 1:1 ===")
v11 = next((v for v in verses if v['chapter']==1 and v['verse']=='1'), None)
if v11:
    print(f"  heb: {clean_hebrew(v11['hebrew'])[:80]}")
    print(f"  eng: {v11['english'][:80]}")

print("\n=== James 1:21 (empty) ===")
v121 = next((v for v in verses if v['chapter']==1 and v['verse']=='21'), None)
if v121:
    print(f"  empty={v121['empty']}  heb='{v121['hebrew'][:40]}'")
else:
    print("  NOT FOUND")

print("\n=== Alt-numbered sample ===")
for v in verses:
    if v['alt_v']:
        print(f"  Jas {v['chapter']}:{v['verse']}  alt {v['alt_ch']}:{v['alt_v']}")

print("\n=== XML well-formedness check ===")
for tree, filename in outputs:
    try:
        etree.tostring(tree, encoding="unicode")
        print(f"  {filename}: OK")
    except Exception as e:
        print(f"  {filename}: ERROR — {e}")